# ДЗ №5. TIGER-like архитектура для рекомендаций

Общая информация  

Дата выдачи: 25 мая 2026

Дедлайн: 10 июня 2026 23:59 MSK


В этом задании мы соберем упрощенный generative retrieval pipeline по мотивам [**TIGER**](https://proceedings.neurips.cc/paper_files/paper/2023/file/20dcab0f14046a5c6b02b61da9f13229-Paper-Conference.pdf):

1. Возьмем датасет **KION**.
2. Построим для айтемов **semantic ids** через **RQ-VAE**.
3. Поверх semantic ids обучим **tiger-like** next-token recommender.
4. Посчитаем метрики на **настоящих `item_id`**, а не на токенах semantic id.
5. Сравним результат с `MostPop` и `SASRec-like` baseline.
6. Отдельно проверим, что происходит, если semantic id обучаются **без уникализирующего suffix/id**, а затем декодируются обратно в item разными способами.

В отличие от классических retrieval-моделей, здесь модель не ранжирует весь каталог напрямую, а **генерирует код следующего айтема**. Поэтому в этом ДЗ важно аккуратно разделять:

- качество самих semantic ids;
- качество generative модели по semantic ids;
- качество итоговых рекомендаций после обратного декодинга в реальные `item_id`.


В этом задании следующая разбалловка:

1. Обучение и анализ semantic ids через `RQ-VAE` - **3 балла**
2. Обучение `tiger-like` next-token модели - **3 балла**
3. Сравнение с `MostPop` и `SASRec-like` - **2 балла**
4. Эксперимент с неуникальными semantic ids и разными стратегиями декодинга - **2 балла**

Соответственно, максимум можно набрать **10 баллов**.


## Короткая теория

### 1. Semantic IDs

Проблема обычных `item_id` в том, что они, как правило, **не несут семантики**: два похожих фильма имеют совершенно независимые идентификаторы.  
Идея semantic ids состоит в том, чтобы сопоставить каждому айтему **короткую дискретную последовательность токенов**, где близкие по смыслу айтемы имеют похожие коды.

Для этого удобно взять фиксированные item-embeddings по контенту и затем **квантовать** их через `RQ-VAE`:

- encoder переводит эмбеддинг в другое скрытое представление;
- residual quantization по нескольким codebook-уровням превращает в последовательность дискретных кодов;
- decoder пытается восстановить исходный embedding;
- дополнительные регуляризаторы позволяют контролировать загрузку codebook-ов, коллизии и энтропию.

### 2. TIGER-like generative retrieval

В статье TIGER recommendation рассматривается как **autoregressive generation** semantic id следующего айтема.  
История пользователя кодируется как последовательность токенов прошлых semantic ids, а модель предсказывает токены следующего айтема.

Упрощенно это выглядит так:

- у айтема есть semantic id, например `[c1, c2, c3, c4]`;
- история пользователя превращается в длинную token-sequence;
- decoder-only Transformer предсказывает следующий токен;
- после генерации semantic id он декодируется обратно в реальный `item_id`.

### 3. Зачем нужен уникализирующий suffix

Если несколько айтемов попадают в один и тот же semantic bucket, возникает **коллизия**.  
Один способ справиться с этим - добавить к semantic id дополнительный уникализирующий suffix.  
Другой путь - оставить id чисто семантическим, а потом восстанавливать `item_id` через отдельный mapping внутри bucket-а. Именно это вы исследуете в последней части ДЗ.


[Ссылка](https://proceedings.neurips.cc/paper_files/paper/2023/file/20dcab0f14046a5c6b02b61da9f13229-Paper-Conference.pdf) на работу по Semantic-ids и TIGER: 




In [ ]:
!pip install -q pandas numpy scikit-learn matplotlib seaborn torch tqdm sentencepiece


In [ ]:
import math
import random
import warnings
from collections import Counter, defaultdict
from dataclasses import dataclass
from pathlib import Path
from typing import Dict, Iterable, List, Optional, Sequence, Tuple

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import torch
import torch.nn as nn
import torch.nn.functional as F
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.decomposition import TruncatedSVD
from sklearn.preprocessing import StandardScaler
from tqdm.auto import tqdm

warnings.filterwarnings("ignore")
sns.set_style("whitegrid")


## 0. Загрузка данных

Работаем с KION-датасетом из соревнования:

https://ods.ai/competitions/competition-recsys-21/data

Нужны файлы:

- `interactions.csv`
- `items.csv`
- `users.csv`

Ноутбук ищет их рядом с собой, уровнем выше или на два уровня выше.


In [ ]:
def find_file(candidates: List[str]) -> Optional[Path]:
    search_roots = [Path("."), Path(".."), Path("../.."), Path("/mnt/data")]
    for root in search_roots:
        for name in candidates:
            path = root / name
            if path.exists():
                return path.resolve()
    return None


interactions_path = find_file(["interactions.csv", "kion_interactions.csv"])
items_path = find_file(["items.csv", "kion_items.csv"])
users_path = find_file(["users.csv", "kion_users.csv"])

print("interactions_path =", interactions_path)
print("items_path        =", items_path)
print("users_path        =", users_path)


In [ ]:
assert interactions_path is not None, "Не найден interactions.csv"
assert items_path is not None, "Не найден items.csv"
assert users_path is not None, "Не найден users.csv"

interactions = pd.read_csv(interactions_path)
items = pd.read_csv(items_path)
users = pd.read_csv(users_path)

print(interactions.shape, items.shape, users.shape)
interactions.head()


## 1. Подготовка выборки

Чтобы не делать ДЗ неподъемным, возьмем **сэмпл активных пользователей и популярных айтемов**.  
При желании вы можете изменить размеры выборки, но в отчете обязательно укажите, какой сэмпл использовали.


In [ ]:
RANDOM_STATE = 42
SAMPLE_USERS = 15000
SAMPLE_ITEMS = 20000
MIN_USER_INTERACTIONS = 5
MAX_SEQ_LEN = 50
TOP_K = 10

random.seed(RANDOM_STATE)
np.random.seed(RANDOM_STATE)
torch.manual_seed(RANDOM_STATE)

TARGET_COL = "item_id"
USER_COL = "user_id"
TIME_COL = "last_watch_dt" if "last_watch_dt" in interactions.columns else "watched_pct"


In [ ]:
interactions = interactions.copy()

if "last_watch_dt" in interactions.columns:
    interactions["last_watch_dt"] = pd.to_datetime(interactions["last_watch_dt"])
    sort_cols = [USER_COL, "last_watch_dt"]
else:
    sort_cols = [USER_COL]

if "total_dur" in interactions.columns:
    interactions["total_dur"] = interactions["total_dur"].fillna(0)

user_activity = interactions[USER_COL].value_counts()
item_popularity = interactions[TARGET_COL].value_counts()

selected_users = user_activity.head(SAMPLE_USERS).index
selected_items = item_popularity.head(SAMPLE_ITEMS).index

df = interactions[
    interactions[USER_COL].isin(selected_users) &
    interactions[TARGET_COL].isin(selected_items)
].copy()

df = df.sort_values(sort_cols).reset_index(drop=True)
df = df[df.groupby(USER_COL)[TARGET_COL].transform("size") >= MIN_USER_INTERACTIONS].copy()

print(df.shape)
df.head()


Сделайте split по пользователям в стиле leave-last-out:

- `train`: вся история кроме двух последних событий пользователя;
- `valid`: предпоследний айтем;
- `test`: последний айтем.

Если хотите, можете использовать другой аккуратный temporal split, но тогда явно опишите его в отчете.


In [ ]:
def leave_last_two_out(data: pd.DataFrame, user_col: str, item_col: str, sort_cols: List[str]):
    data = data.sort_values(sort_cols).copy()
    data["_row_num"] = data.groupby(user_col).cumcount()
    data["_user_len"] = data.groupby(user_col)[item_col].transform("size")

    train = data[data["_row_num"] < data["_user_len"] - 2].copy()
    valid = data[data["_row_num"] == data["_user_len"] - 2].copy()
    test = data[data["_row_num"] == data["_user_len"] - 1].copy()

    return (
        train.drop(columns=["_row_num", "_user_len"]),
        valid.drop(columns=["_row_num", "_user_len"]),
        test.drop(columns=["_row_num", "_user_len"]),
    )


train_df, valid_df, test_df = leave_last_two_out(df, USER_COL, TARGET_COL, sort_cols)

print(train_df.shape, valid_df.shape, test_df.shape)


In [ ]:
train_sequences = (
    train_df
    .sort_values(sort_cols)
    .groupby(USER_COL)[TARGET_COL]
    .apply(list)
    .to_dict()
)

valid_targets = dict(zip(valid_df[USER_COL], valid_df[TARGET_COL]))
test_targets = dict(zip(test_df[USER_COL], test_df[TARGET_COL]))

eval_users = sorted(set(test_targets) & set(train_sequences))
len(eval_users)


## 2. Метрики

Мы сравниваем модели уже на **реальных item_id**.  
Реализуйте (или переиспользуйте реализации из предыдущих ДЗ) как минимум:

- `Recall@K`
- `NDCG@K`
- `MRR@K`

Можно дополнительно посчитать `HitRate@K`, `Coverage@K`, `Novelty@K`.


In [ ]:
def recall_at_k(recs: Dict[int, List[int]], target: Dict[int, int], k: int = 10) -> float:
    # YOUR CODE HERE
    raise NotImplementedError


def ndcg_at_k(recs: Dict[int, List[int]], target: Dict[int, int], k: int = 10) -> float:
    # YOUR CODE HERE
    raise NotImplementedError


def mrr_at_k(recs: Dict[int, List[int]], target: Dict[int, int], k: int = 10) -> float:
    # YOUR CODE HERE
    raise NotImplementedError


def evaluate_recommendations(recs: Dict[int, List[int]], target: Dict[int, int], k: int = 10) -> pd.DataFrame:
    return pd.DataFrame(
        {
            f"Recall@{k}": [recall_at_k(recs, target, k)],
            f"NDCG@{k}": [ndcg_at_k(recs, target, k)],
            f"MRR@{k}": [mrr_at_k(recs, target, k)],
        },
        index=["score"],
    )


## I. Semantic IDs через RQ-VAE (3 балла)

В этой части нужно:

1. Построить фиксированные item-embeddings по метаданным айтемов.
2. Обучить `RQ-VAE`, получив многотокенный semantic id.
3. Посчитать и визуализировать метрики качества semantic ids.

Обязательно покажите графики обучения хотя бы по:

- reconstruction loss;
- доле коллизий;
- энтропии использования кодов;
- average bucket size или effective codebook usage.

Дополнительно приветствуются:

- доля неиспользуемых кодов;

Реализацию вы можете взять отсюда [https://github.com/EdoardoBotta/RQ-VAE-Recommender](https://github.com/EdoardoBotta/RQ-VAE-Recommender)


Для простоты можно собрать content features из `title`, `genres`, `countries`, `release_year`, других доступных полей.  
Необязательно делать сильный content encoder: в этой работе главное не качество текстовой модели, а корректный пайплайн `semantic ids -> generative retrieval`.

Ниже используется TF-IDF, но вы можете заменить эмбеддер на любую хорошую языковую модель


In [ ]:
def split_tokens(x) -> List[str]:
    if pd.isna(x):
        return []
    text = str(x)
    for sep in [",", "|", "/"]:
        if sep in text:
            return [part.strip() for part in text.split(sep) if part.strip()]
    return [text.strip()] if text.strip() else []


items_work = items[items[TARGET_COL].isin(pd.Index(df[TARGET_COL].unique()))].copy()
items_work["title"] = items_work.get("title", pd.Series("", index=items_work.index)).fillna("")
items_work["genres_text"] = items_work.get("genres", pd.Series("", index=items_work.index)).fillna("")
items_work["countries_text"] = items_work.get("countries", pd.Series("", index=items_work.index)).fillna("")
items_work["release_year_num"] = pd.to_numeric(
    items_work.get("release_year", pd.Series(np.nan, index=items_work.index)),
    errors="coerce",
).fillna(0)

items_work["content_text"] = (
    items_work["title"].astype(str) + " " +
    items_work["genres_text"].astype(str) + " " +
    items_work["countries_text"].astype(str)
)

items_work = items_work.drop_duplicates(subset=[TARGET_COL]).reset_index(drop=True)
items_work.head()


In [ ]:
vectorizer = TfidfVectorizer(max_features=10000, ngram_range=(1, 2), min_df=2)
text_features = vectorizer.fit_transform(items_work["content_text"])

svd = TruncatedSVD(n_components=128, random_state=RANDOM_STATE)
text_dense = svd.fit_transform(text_features)

num_features = items_work[["release_year_num"]].to_numpy(dtype=np.float32)
num_features = StandardScaler().fit_transform(num_features)

item_content_matrix = np.concatenate([text_dense, num_features], axis=1).astype(np.float32)
item_ids = items_work[TARGET_COL].tolist()

print(item_content_matrix.shape)


In [ ]:
@dataclass
class RQVAEConfig:
    input_dim: int
    hidden_dim: int = 256
    latent_dim: int = 64
    num_codebooks: int = 4
    codebook_size: int = 64
    commitment_weight: float = 0.25
    entropy_weight: float = 0.0


class RQVAE(nn.Module):
    def __init__(self, config: RQVAEConfig):
        super().__init__()
        self.config = config
        # YOUR CODE HERE

    def encode(self, x: torch.Tensor):
        # YOUR CODE HERE
        raise NotImplementedError

    def quantize(self, z_e: torch.Tensor):
        # YOUR CODE HERE
        raise NotImplementedError

    def decode(self, z_q: torch.Tensor):
        # YOUR CODE HERE
        raise NotImplementedError

    def forward(self, x: torch.Tensor):
        # YOUR CODE HERE
        raise NotImplementedError


In [ ]:
def semantic_collision_rate(codes: np.ndarray) -> float:
    unique_codes = {tuple(row.tolist()) for row in codes}
    return 1.0 - len(unique_codes) / len(codes)


def average_bucket_size(codes: np.ndarray) -> float:
    buckets = Counter(map(tuple, codes.tolist()))
    return float(np.mean(list(buckets.values())))


def codebook_entropy(codes_2d: np.ndarray, codebook_size: int) -> List[float]:
    entropies = []
    for level in range(codes_2d.shape[1]):
        counts = np.bincount(codes_2d[:, level], minlength=codebook_size).astype(np.float64)
        probs = counts / counts.sum()
        probs = probs[probs > 0]
        entropies.append(float(-(probs * np.log(probs + 1e-12)).sum()))
    return entropies


In [ ]:
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
DEVICE


In [ ]:
rq_config = RQVAEConfig(input_dim=item_content_matrix.shape[1])
rqvae = RQVAE(rq_config).to(DEVICE)

# YOUR CODE HERE
# 1. соберите DataLoader
# 2. обучите модель
# 3. сохраняйте историю метрик по эпохам:
#    - total loss
#    - reconstruction loss
#    - commitment / vq loss
#    - collision rate
#    - entropy per codebook
#    - average bucket size


In [ ]:
# YOUR CODE HERE
# Постройте графики обучения semantic ids


In [ ]:
# YOUR CODE HERE
# После обучения получите semantic ids для всех айтемов
# Желательно сохранить:
# - item_id
# - semantic_tokens (без уникализатора)
# - semantic_tokens_unique (с уникализатором, если вы его добавляете)


Сформируйте итоговую таблицу по semantic ids. Например:

- число уникальных semantic ids;
- collision rate;
- средняя мощность bucket-а;
- средняя энтропия по codebook-уровням;
- число неиспользованных кодов по уровням.


In [ ]:
# YOUR CODE HERE
semantic_metrics_table = pd.DataFrame()
semantic_metrics_table


По результатам этого пункта у вас должна получиться таблица: item_id -> semantic_id, причем должны быть все айтемы помапплены. Еще раз убедитесь, что вы понимаете проблему коллизий. Если RQ-VAE кодирует 3 кодбука, то s-id = 3 кода + 1 уникализирующий код. Вместе с уникализириющим кодом s-id становится полноценным (однозначным) идентификатором конкретного айтема. Без этого кода вы теряете часть информации.

Далее вы можете обучать модель как с этим уникализирующим кодом, так и без него. Однако здесь у вас будет возникать разница:
1) если TIGER работает без уникализирующего id в конце, то вам надо дополнительно сделать процедуру декодинга. Тогда TIGER работает только на семантике, а рекомендацией из semantic в конкретные айди занимаетесь уже вы сами. TIGER также теряет часть информации, но с другой стороны фокусируется только на контентной составляющей.
2) если TIGER работает с уникализирующим id в конце, то подразумевается, что модель будет рекомендовать вам полноценные sid, эквивалентные item_id (с маппингом 1 к 1). В этом случае модель будет уже не только на семантической информации, но и в том числе ей надо будет запоминать конкретные айдишники (это более сложная задача)

В последнем пункте (если вы хотите получить максимальный балл) вам в любом случае надо будет сравнить оба подхода, поэтому выбранный тут подход остается на ваше усмотрение


## II. Tiger-like next-token prediction (3 балла)

Теперь построим `tiger-like` модель поверх semantic ids.

Нужно:

1. Преобразовать пользовательские истории в последовательности semantic-токенов.
2. Обучить causal next-token модель (например, GPT2 from scratch), можно не очень большую (4/4/256, например)
3. Сгенерировать рекомендации.
4. Декодировать сгенерированные semantic ids обратно в `item_id`.
5. Посчитать финальные метрики на настоящих айтемах.


Важно: качество оценивается не по совпадению токенов само по себе, а по тому, удается ли после декодинга получить релевантные **реальные** `item_id`.


In [ ]:
SPECIAL_TOKENS = {
    "PAD": 0,
    "BOS": 1,
    "EOS": 2,
    "SEP": 3,
}

# YOUR CODE HERE
# 1. Постройте общий словарь semantic tokens
# 2. Напишите mapping item_id -> token sequence
# 3. Подготовьте train / valid / test sequences для autoregressive learning


In [ ]:
class TigerSequenceDataset(torch.utils.data.Dataset):
    def __init__(self, samples: List[Dict[str, torch.Tensor]]):
        self.samples = samples

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        return self.samples[idx]


def collate_batch(batch):
    # YOUR CODE HERE
    raise NotImplementedError


In [ ]:
class TigerLikeModel(nn.Module):
    def __init__(
        self,
        vocab_size: int,
        d_model: int = 192,
        n_heads: int = 4,
        n_layers: int = 3,
        dropout: float = 0.1,
        max_len: int = 512,
    ):
        super().__init__()
        self.vocab_size = vocab_size
        self.d_model = d_model
        # YOUR CODE HERE

    def forward(self, input_ids: torch.Tensor, attention_mask: Optional[torch.Tensor] = None):
        # YOUR CODE HERE
        raise NotImplementedError


In [ ]:
tiger_model = TigerLikeModel(vocab_size=0).to(DEVICE)  # замените vocab_size

# YOUR CODE HERE
# Напишите обучение:
# - teacher forcing
# - next-token cross-entropy
# - валидация на valid split


In [ ]:
def decode_semantic_sequence(
    token_ids: List[int],
    sid_to_items: Dict[Tuple[int, ...], List[int]],
    strategy: str = "top1_pop",
    item_popularity: Optional[Dict[int, int]] = None,
    seen_items: Optional[set] = None,
) -> Optional[int]:
    # YOUR CODE HERE
    # strategy может быть, например:
    # - top1_pop
    # - top2_pop
    # - top3_pop
    # - random
    # - all
    raise NotImplementedError


def generate_tiger_recommendations(model, users_subset: List[int], top_k: int = 10):
    # YOUR CODE HERE
    # Верните Dict[user_id, List[item_id]]
    raise NotImplementedError


In [ ]:
# YOUR CODE HERE
tiger_recs = {}
tiger_metrics = evaluate_recommendations(tiger_recs, test_targets, k=TOP_K)
tiger_metrics


Сделайте небольшую качественную таблицу с рекомендациями. Например:

- `user_id`
- последние 5 просмотренных айтемов
- предсказанный `semantic_id`
- декодированный `item_id`
- title декодированного айтема

Это удобно для sanity-check: модель может выдавать хорошие токены, но декодинг обратно в `item_id` работать плохо.


In [ ]:
# YOUR CODE HERE
recommendation_examples = pd.DataFrame()
recommendation_examples.head(10)


## III. Baselines: MostPop и SASRec-like (2 балла)

Теперь сравните `tiger-like` с двумя baseline-подходами:

1. `MostPop`
2. `SASRec-like`

Важно, чтобы все модели оценивались на одном и том же `test`-split и по одним и тем же метрикам.


In [ ]:
class MostPopRecommender:
    def __init__(self):
        self.popular_items = []

    def fit(self, data: pd.DataFrame):
        self.popular_items = data[TARGET_COL].value_counts().index.tolist()
        return self

    def predict(self, user_ids: Iterable[int], top_k: int = 10) -> Dict[int, List[int]]:
        return {user_id: self.popular_items[:top_k] for user_id in user_ids}


most_pop = MostPopRecommender().fit(train_df)
most_pop_recs = most_pop.predict(eval_users, top_k=TOP_K)
most_pop_metrics = evaluate_recommendations(most_pop_recs, test_targets, k=TOP_K)
most_pop_metrics


In [ ]:
class SASRecLikeDataset(torch.utils.data.Dataset):
    def __init__(self, user_sequences: Dict[int, List[int]], max_len: int = 50):
        self.user_ids = sorted(user_sequences)
        self.user_sequences = user_sequences
        self.max_len = max_len

    def __len__(self):
        return len(self.user_ids)

    def __getitem__(self, idx):
        user_id = self.user_ids[idx]
        seq = self.user_sequences[user_id][-self.max_len:]
        # YOUR CODE HERE
        raise NotImplementedError


In [ ]:
class SASRecLikeModel(nn.Module):
    def __init__(
        self,
        num_items: int,
        d_model: int = 128,
        n_heads: int = 4,
        n_layers: int = 2,
        max_len: int = 50,
        dropout: float = 0.1,
    ):
        super().__init__()
        # YOUR CODE HERE

    def forward(self, input_ids: torch.Tensor):
        # YOUR CODE HERE
        raise NotImplementedError


In [ ]:
# YOUR CODE HERE
# 1. обучите SASRec-like
# 2. получите рекомендации
sasrec_recs = {}
sasrec_metrics = evaluate_recommendations(sasrec_recs, test_targets, k=TOP_K)
sasrec_metrics


In [ ]:
comparison_table = pd.concat(
    [
        most_pop_metrics.rename(index={"score": "MostPop"}),
        sasrec_metrics.rename(index={"score": "SASRec-like"}),
        tiger_metrics.rename(index={"score": "TIGER-like"}),
    ]
)

comparison_table


В отчете обязательно прокомментируйте:

- где `TIGER-like` выигрывает или проигрывает `SASRec-like`;
- насколько сильно отстает `MostPop`;
- как качество semantic ids связано с итоговым recommendation quality.


## IV. Эксперимент без unique suffix и с разными стратегиями декодинга (2 балла)

Повторите эксперимент в варианте, где semantic id **не уникализируется**.

Тогда внутри одного semantic bucket может находиться несколько реальных айтемов.  
Нужно сравнить несколько способов декодинга bucket -> `item_id`, например:

- `top1_pop` - рекомендуем топ-1 айтем внутри одного sid
- `top2_pop` - рекомендуем топ-2 айтема внутри одного sid
- `top3_pop` - рекомендуем топ-3 айтемов внутри одного sid
- `random`
- `all`

То есть тут у вас TIGER работает чисто на семантике, а вы реализуете разные стратегии декодинга и сравниваете их


In [ ]:
decoding_results = []

for strategy in ["top1_pop", "top2_pop", "top3_pop", "random", "all"]:
    # YOUR CODE HERE
    # 1. постройте рекомендации после декодинга
    # 2. посчитайте метрики
    # 3. добавьте строку в decoding_results
    pass

decoding_table = pd.DataFrame(decoding_results)
decoding_table


Отдельно сравните:

- качество с unique suffix;
- качество без unique suffix;
- чувствительность к выбранной стратегии декодинга.

И также напишите общий вывод по проделанной работе


In [ ]:
# YOUR CODE HERE
# Дополнительные графики / анализ
